In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

CITY_COUNTRY = {"Amsterdam": "NLD", "Barcelona": "ESP", "Berlin": "DEU", "Columbus": "USA"}
COUNTRY_CITY = {v: k for k, v in CITY_COUNTRY.items()}

In [2]:
wb_raw = pd.read_sql("SELECT * FROM raw_economic_indicators", engine)
print("shape:", wb_raw.shape)
wb_raw.head()

shape: (96, 5)


,country_code,country,year,indicator,value
0,DEU,Germany,2025,gdp_per_capita_usd,60496.435082
1,DEU,Germany,2024,gdp_per_capita_usd,56103.732318
2,DEU,Germany,2023,gdp_per_capita_usd,54776.766824
3,DEU,Germany,2022,gdp_per_capita_usd,50506.517964
4,DEU,Germany,2021,gdp_per_capita_usd,52349.245999


In [3]:
wb = wb_raw.copy()

wb["year"] = pd.to_numeric(wb["year"], errors="coerce").astype("Int64")
wb["value"] = pd.to_numeric(wb["value"], errors="coerce")

n0 = len(wb)
wb = wb.drop_duplicates(subset=["country_code", "indicator", "year"])
print("duplicates removed:", n0 - len(wb))

print("\nmissing values:")
print(wb[["year", "value"]].isna().sum())

duplicates removed: 0

missing values:
year      0
value    21
dtype: int64


In [4]:
econ = (wb.pivot(index=["country_code", "country", "year"],
                 columns="indicator", values="value")
          .reset_index())
econ.columns.name = None

ind_cols = [c for c in econ.columns if c not in ("country_code", "country", "year")]

print("non-missing values per country (out of", econ["year"].nunique(), "years):")
print(econ.groupby("country_code")[ind_cols].count())

print("\nshare missing per indicator:")
print(econ[ind_cols].isna().mean().round(2))

non-missing values per country (out of 8 years):
              gdp_per_capita_usd  inflation_pct  \
country_code                                      
DEU                            8              8   
ESP                            8              8   
NLD                            8              8   
USA                            8              7   

              international_tourism_arrivals  
country_code                                  
DEU                                        3  
ESP                                        3  
NLD                                        3  
USA                                        3  

share missing per indicator:
gdp_per_capita_usd                0.00
inflation_pct                     0.03
international_tourism_arrivals    0.62
dtype: float64


In [5]:
miss = econ[ind_cols].isna().mean()
too_empty = miss[miss > 0.5].index.tolist()
print("dropped (more than 50% missing):", too_empty)

economy_clean = econ.drop(columns=too_empty)
economy_clean.insert(0, "city", economy_clean["country_code"].map(COUNTRY_CITY))
economy_clean = economy_clean.sort_values(["city", "year"]).reset_index(drop=True)

print(economy_clean.shape)
economy_clean.head(8)

dropped (more than 50% missing): ['international_tourism_arrivals']
(32, 6)


,city,country_code,country,year,gdp_per_capita_usd,inflation_pct
0,Amsterdam,NLD,Netherlands,2018,53955.077002,1.703619
1,Amsterdam,NLD,Netherlands,2019,53554.900749,2.633974
2,Amsterdam,NLD,Netherlands,2020,53467.927741,1.267881
3,Amsterdam,NLD,Netherlands,2021,60141.988091,2.681233
4,Amsterdam,NLD,Netherlands,2022,59123.318556,9.998173
5,Amsterdam,NLD,Netherlands,2023,63515.603078,3.842320
6,Amsterdam,NLD,Netherlands,2024,67465.320813,3.346399
7,Amsterdam,NLD,Netherlands,2025,73683.917133,3.203640


In [6]:
economy_clean.to_csv(OUT / "economy_clean.csv", index=False)
economy_clean.to_parquet(OUT / "economy_clean.parquet", index=False)
print("saved:", sorted(p.name for p in OUT.iterdir()))

saved: ['calendar_daily_city.csv', 'calendar_listing_summary.csv', 'economy_clean.csv', 'economy_clean.parquet', 'listings_clean.csv', 'weather_clean.csv', 'weather_clean.parquet']
